# Goodreads Bookshelf Project
As part of a personal project, I wanted to gather my Goodreads data and explore how many books (and pages) I have
read in a year, who my most read authors are and how many authors in general I've read, the genres I
gravitated toward, how many pages I read per day during specific years, how long it took me to read a book,
and my average rating for books.

In [1]:
import re
import importlib
import requests
import datetime
import json
from pathlib import Path

from tqdm import tqdm
import pandas as pd
import numpy as np

import lib


importlib.reload(lib)

log = lib.getLogger('goodreads_dataset')
working_directory = Path().cwd()
export_clean_directory = working_directory / 'clean'

In [2]:
path = '/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/goodreads_library_export.csv'

## Read in the goodreads data

In [3]:
df = lib.read_data(path)

PortfolioLogger.lib.tools: INFO: Encoding: utf-8
PortfolioLogger.lib.tools: INFO: Stripping whitespaces from goodreads_library_export.csv
PortfolioLogger.lib.performance: INFO: <function read_data at 0x10ef5a520> took 0.031 secs to complete.


# Checking out the data

In [4]:
df.head()

,Book Id,Title,Author,Author l-f,Additional Authors,ISBN,ISBN13,My Rating,Average Rating,Publisher,...,Date Read,Date Added,Bookshelves,Bookshelves with positions,Exclusive Shelf,My Review,Spoiler,Private Notes,Read Count,Owned Copies
0,18400112,"The Devil Wears Scrubs (Dr. Jane McGill, #1)",Freida McFadden,"McFadden, Freida",NaN,"=""""","=""""",3,3.44,Hollywood Upstairs Publishing,...,NaN,2025/12/01,NaN,NaN,read,NaN,NaN,NaN,1,0
1,62047984,Yellowface,R.F. Kuang,"Kuang, R.F.",NaN,"=""""","=""""",4,3.73,William Morrow,...,NaN,2023/12/10,NaN,NaN,read,NaN,NaN,NaN,1,0
2,58416952,"The Will of the Many (Hierarchy, #1)",James Islington,"Islington, James",NaN,"=""1982141190""","=""9781982141196""",0,4.60,Gallery / Saga Press,...,NaN,2025/12/01,to-read,to-read (#52),to-read,NaN,NaN,NaN,0,0
3,20886354,"Skin Deep (Legion, #2)",Brandon Sanderson,"Sanderson, Brandon",Jon Foster,"=""""","=""""",4,4.12,Subterranean Press,...,2025/11/07,2025/11/07,NaN,NaN,read,NaN,NaN,NaN,1,0
4,58778536,Do Not Disturb,Freida McFadden,"McFadden, Freida",NaN,"=""""","=""""",3,3.90,Hollywood Upstairs Publishing,...,2025/11/13,2025/11/18,NaN,NaN,read,NaN,NaN,NaN,1,0


# Creating helper functions to clean ISBN code
They're currently stored as ="" or ="isbn#"

In [9]:
def clean_isbn(isbn):
    return re.sub(r'["=]', '', isbn)

# There were no genres stored within the Goodreads dataset so I'l be using GoogleAPI and Open AI to gather genre data for the books using the ISBN13 book code.

In [12]:
def request_openlibrary_book_data(isbn):
    isbn = clean_isbn(isbn)
    url = f"https://openlibrary.org/isbn/{isbn}.json"
    response = requests.get(url)
    if not response:
        return {}

    response = response.json()

    # Get book "works" metadata (contains subjects)
    works_key = response.get("works", [{}])[0].get("key")
    if not works_key:
        return {}

    works_data = requests.get(f"https://openlibrary.org{works_key}.json").json()
    return works_data
request_openlibrary_book_data(df.loc[2, 'ISBN13'])

{'type': {'key': '/type/work'},
 'title': 'The Will of the Many',
 'authors': [{'author': {'key': '/authors/OL7468631A'},
   'type': {'key': '/type/author_role'}}],
 'key': '/works/OL31088394W',
 'subjects': ['series:Hierarchy', 'genre:high fantasy'],
 'description': "The Catenan Republic—the Hierarchy—may rule the world, but they do not know everything.\r\n\r\nI tell them my name is Vis Telimus. I tell them I was orphaned three years ago, and that only good fortune has got me into their most prestigious school. I tell them that, when I graduate, I will allow my strength and drive—what they call Will—to be leeched away and added to the power of those above me, as everyone must do.\r\n\r\nI tell them that I belong, and they believe me.\r\n\r\nBut the truth is that I have been sent to the Academy to solve a murder. To search for an ancient weapon. To uncover secrets that may tear the Republic apart.\r\n\r\nAnd that I will never cede my Will to the empire that executed my family.\r\n\r\nT

In [13]:
def request_googleapi_book_data(isbn):
    isbn = clean_isbn(isbn)
    url = f"https://www.googleapis.com/books/v1/volumes?q=isbn:{isbn}"
    response = requests.get(url).json()
    if not 'items' in response:
        return {}
    return response['items'][0]
request_googleapi_book_data(df.loc[2, 'ISBN13'])

{'kind': 'books#volume',
 'id': 'jlGUEAAAQBAJ',
 'etag': 'mtUK79ne1mk',
 'selfLink': 'https://www.googleapis.com/books/v1/volumes/jlGUEAAAQBAJ',
 'volumeInfo': {'title': 'The Will of the Many',
  'authors': ['James Islington'],
  'publisher': 'Simon and Schuster',
  'publishedDate': '2023-05-23',
  'description': 'At the elite Catenan Academy, a young fugitive uncovers layered mysteries and world-changing secrets in this “brilliant and gut-churning masterpiece” (Library Journal, starred review) by the internationally bestselling author of The Licanius Trilogy, James Islington. The Catenan Republic—the Hierarchy—may rule the world now, but they do not know everything. I tell them my name is Vis Telimus. I tell them I was orphaned after a tragic accident three years ago, and that good fortune alone has led to my acceptance into their most prestigious school. I tell them that once I graduate, I will gladly join the rest of civilized society in allowing my strength, my drive, and my focus—

## Data that's useful in the gathered data from GoodleAPI and Open AI.
- volumeInfo
    - authors
    - title
    - publisher
    - publishDate
    - description
    - pageCount
    - categories [list]
    - maturityRating
    - imagelinks (do i want images?): [dict]
    - saleInfo
        - country
        - retailPrice
- subjects

In [7]:
def clean_categories_list(categories, ignore=('collectionID', 'nyt')):
    new_categories = set()
    if not categories:
        return list(new_categories)
    for i in range(len(categories)):
        for category in categories[i].split(', '):
            found = False
            for item in ignore:
                if item in category:
                    found = True
                    break
            if found:
                continue
            new_categories.add(category.title().replace('_', ' '))

    return list(new_categories)


## Final function to organize data for categories (genres)

In [15]:
def get_book_data(isbn):
    """

    """
    isbn = clean_isbn(isbn)

    result = {'Categories': []}

    openai_categories_data = request_openlibrary_book_data(isbn).get('subjects', [])
    googleapi_data = request_googleapi_book_data(isbn)
    if googleapi_data:
        result.update(
            {'Published Date': googleapi_data['volumeInfo'].get('publishedDate'),
            'Categories': clean_categories_list(googleapi_data['volumeInfo'].get('categories')),
            'Maturity Rating': googleapi_data['volumeInfo'].get('maturityRating'),
            'Description': googleapi_data['volumeInfo'].get('description'),
            'Country': googleapi_data['saleInfo'].get('country'),
            'Retail Price': googleapi_data['saleInfo'].get('retailPrice', {}).get('amount'),
            'Currency': googleapi_data['saleInfo'].get('retailPrice', {}).get('currencyCode')}
        )
    if openai_categories_data:
        result['Categories'].extend(clean_categories_list(openai_categories_data))
    result['Categories'] = sorted(set((result['Categories'])))

    return result


## Sample request to see what data I get

In [16]:
data = get_book_data(df.loc[2, 'ISBN13'])
data

{'Categories': ['Fiction', 'Genre:High Fantasy', 'Series:Hierarchy'],
 'Published Date': '2023-05-23',
 'Maturity Rating': 'NOT_MATURE',
 'Description': 'At the elite Catenan Academy, a young fugitive uncovers layered mysteries and world-changing secrets in this “brilliant and gut-churning masterpiece” (Library Journal, starred review) by the internationally bestselling author of The Licanius Trilogy, James Islington. The Catenan Republic—the Hierarchy—may rule the world now, but they do not know everything. I tell them my name is Vis Telimus. I tell them I was orphaned after a tragic accident three years ago, and that good fortune alone has led to my acceptance into their most prestigious school. I tell them that once I graduate, I will gladly join the rest of civilized society in allowing my strength, my drive, and my focus—what they call Will—to be leeched away and added to the power of those above me, as millions already do. As all must eventually do. I tell them that I belong, and

# Add the extra data into the DataFrame

In [13]:
df.columns

Index(['Book Id', 'Title', 'Author', 'Author l-f', 'Additional Authors',
       'ISBN', 'ISBN13', 'My Rating', 'Average Rating', 'Publisher', 'Binding',
       'Number of Pages', 'Year Published', 'Original Publication Year',
       'Date Read', 'Date Added', 'Bookshelves', 'Bookshelves with positions',
       'Exclusive Shelf', 'My Review', 'Spoiler', 'Private Notes',
       'Read Count', 'Owned Copies'],
      dtype='object')

In [17]:
categories = []
concat_df = pd.DataFrame(df['Book Id'])
df_categories = pd.DataFrame()
for idx, row in tqdm(df.iterrows(), total=len(df)):
    isbn = clean_isbn(row['ISBN13'])
    book_data = get_book_data(isbn)
    if book_data:
        for key, value in book_data.items():
            if key == 'Categories':
                for category in book_data['Categories']:
                    categories.append(
                        {'Book Id': row['Book Id'], 'Category': category}
                    )
                continue
            concat_df.at[idx, key] = value

concat_df

100%|██████████| 237/237 [03:03<00:00,  1.29it/s]


,Book Id,Published Date,Maturity Rating,Description,Country,Retail Price,Currency
0,18400112,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
1,62047984,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
2,58416952,2023-05-23,NOT_MATURE,"At the elite Catenan Academy, a young fugitive...",US,16.99,USD
3,20886354,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
4,58778536,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
...,...,...,...,...,...,...,...
232,40389527,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
233,38389488,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
234,39863488,2019-02-05,NOT_MATURE,"INSTANT NEW YORK TIMES BESTSELLER ""I love Jane...",US,None,None
235,43848929,2019-09-10,NOT_MATURE,"Malcolm Gladwell, host of the podcast Revision...",US,None,None


In [18]:
df_cp1 = df.merge(concat_df, how='left', on='Book Id')

In [20]:
df_cp1.head()

,Book Id,Title,Author,Author l-f,Additional Authors,ISBN,ISBN13,My Rating,Average Rating,Publisher,...,Spoiler,Private Notes,Read Count,Owned Copies,Published Date,Maturity Rating,Description,Country,Retail Price,Currency
0,18400112,"The Devil Wears Scrubs (Dr. Jane McGill, #1)",Freida McFadden,"McFadden, Freida",NaN,"=""""","=""""",3,3.44,Hollywood Upstairs Publishing,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
1,62047984,Yellowface,R.F. Kuang,"Kuang, R.F.",NaN,"=""""","=""""",4,3.73,William Morrow,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
2,58416952,"The Will of the Many (Hierarchy, #1)",James Islington,"Islington, James",NaN,"=""1982141190""","=""9781982141196""",0,4.60,Gallery / Saga Press,...,NaN,NaN,0,0,2023-05-23,NOT_MATURE,"At the elite Catenan Academy, a young fugitive...",US,16.99,USD
3,20886354,"Skin Deep (Legion, #2)",Brandon Sanderson,"Sanderson, Brandon",Jon Foster,"=""""","=""""",4,4.12,Subterranean Press,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
4,58778536,Do Not Disturb,Freida McFadden,"McFadden, Freida",NaN,"=""""","=""""",3,3.90,Hollywood Upstairs Publishing,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None


# Create a new Dataframe for the categories

In [32]:
df_categories = pd.DataFrame(categories)
df_categories

,Book Id,Category
0,18400112,Language Arts & Disciplines
1,62047984,Language Arts & Disciplines
2,58416952,Fiction
3,58416952,Genre:High Fantasy
4,58416952,Series:Hierarchy
...,...,...
3769,43848929,Social Science
3770,43848929,Strangers
3771,43848929,Threat (Psychology)
3772,43848929,Trust


## Check genre counts

In [33]:
category_counts = df_categories['Category'].value_counts()
category_counts

Category
Fiction                        129
General                         80
New York Times Bestseller       77
Language Arts & Disciplines     74
Fantasy                         49
                              ... 
Robopsychology                   1
Readers (Secondary)              1
Positronic Brains                1
Hyperspace                       1
Trust                            1
Name: count, Length: 2034, dtype: int64

In [34]:
len(category_counts)

2034

## 2034 categories is too much... let's prune the list
Remove some specific generic genres and any genre that only has 1

In [35]:
categories_to_drop = [
    'Fiction',
    'General',
    'New York Times Bestseller',
    'Language Arts & Disciplines',
    'Romans',
    'Large Type Books',
    'New York Times Reviewed',
    'Reading Level-Grade 11',
    'Reading Level-Grade 12',
    'Open Library Staff Picks',
    'Long Now Manual For Civilization',
    'Ficción',
    'Novela',
    'Reading Level-Grade 10',
    'Reading Level-Grade 9'
] + category_counts[category_counts >= 3].index.tolist()
categories_to_drop

['Fiction',
 'General',
 'New York Times Bestseller',
 'Language Arts & Disciplines',
 'Romans',
 'Large Type Books',
 'New York Times Reviewed',
 'Reading Level-Grade 11',
 'Reading Level-Grade 12',
 'Open Library Staff Picks',
 'Long Now Manual For Civilization',
 'Ficción',
 'Novela',
 'Reading Level-Grade 10',
 'Reading Level-Grade 9',
 'Fiction',
 'General',
 'New York Times Bestseller',
 'Language Arts & Disciplines',
 'Fantasy',
 'Science Fiction',
 'Nouvelles',
 'Romans',
 'Large Type Books',
 'Fantasy Fiction',
 'New York Times Reviewed',
 'Action & Adventure',
 'Magic',
 'English Literature',
 'American Literature',
 'Epic',
 'Reading Level-Grade 11',
 "Children'S Fiction",
 'Reading Level-Grade 12',
 'Open Library Staff Picks',
 'Long Now Manual For Civilization',
 'Novela',
 'Ficción',
 'Psychological',
 'Thrillers',
 'Historical',
 'Reading Level-Grade 10',
 'Literary',
 'Friendship',
 'Imaginary Places',
 'Juvenile Fiction',
 'Survival',
 'Adventure',
 'Good And Evil',
 '

In [36]:
drop_mask = df_categories[df_categories['Category'].isin(categories_to_drop)]

In [37]:
df_categories = df_categories.drop(drop_mask.index, axis=0)

In [38]:
df_categories

,Book Id,Category
3,58416952,Genre:High Fantasy
4,58416952,Series:Hierarchy
17,37640636,Genius
23,13452375,Genius
24,13452375,Gifted Persons
...,...,...
3764,43848929,Language Arts & Disciplines / Communication
3765,43848929,Miscellanea
3770,43848929,Strangers
3771,43848929,Threat (Psychology)


# Cleaning main book Dataframe

In [39]:
df_cp1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Book Id                     237 non-null    int64  
 1   Title                       237 non-null    object 
 2   Author                      237 non-null    object 
 3   Author l-f                  237 non-null    object 
 4   Additional Authors          55 non-null     object 
 5   ISBN                        237 non-null    object 
 6   ISBN13                      237 non-null    object 
 7   My Rating                   237 non-null    int64  
 8   Average Rating              237 non-null    float64
 9   Publisher                   233 non-null    object 
 10  Binding                     237 non-null    object 
 11  Number of Pages             236 non-null    float64
 12  Year Published              236 non-null    float64
 13  Original Publication Year   235 non

In [40]:
for col, series in df_cp1[['ISBN', 'ISBN13']].items():
    df_cp1[col] = df_cp1[col].apply(clean_isbn)

In [41]:
df_cp1['Additional Authors'], _ = lib.fillnull(df_cp1['Additional Authors'], 'None')
lib.error_rate(df_cp1['Additional Authors'])

0.0

In [42]:
df_cp1['Number of Pages'], _ = lib.fillnull(df_cp1['Number of Pages'], 0)
df_cp1['Number of Pages'] = df_cp1['Number of Pages'].astype(int)
lib.error_rate(df_cp1['Number of Pages'])

0.0

## Was replaced with Published Date from GoogleAPI (had yyyy/mm/dd instead of just the year)

In [ ]:
df_cp1 = df_cp1.drop('Year Published', axis=1)

In [45]:
df_cp1['Original Publication Year'], _ = lib.fillnull(df_cp1['Original Publication Year'], 0)
df_cp1['Original Publication Year'] = df_cp1['Original Publication Year'].astype(int)

In [46]:
df_cp1['Date Read'] = pd.to_datetime(df_cp1['Date Read'])
df_cp1['Date Added'] = pd.to_datetime(df_cp1['Date Added'])
df_cp1['Published Date'] = pd.to_datetime(df_cp1['Published Date'], format='mixed')

In [47]:
df_cp1['Retail Price'] = df_cp1['Retail Price'].astype(float)

## Fixing Authors column

In [48]:
to_replace = {
    'Abraham   Verghese': 'Abraham Verghese',
    'Stephen        King': 'Stephen King'
}

for author, fixed in to_replace.items():
    df['Author'] = df['Author'].str.replace(author, fixed)

In [49]:
df_cp1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Book Id                     237 non-null    int64         
 1   Title                       237 non-null    object        
 2   Author                      237 non-null    object        
 3   Author l-f                  237 non-null    object        
 4   Additional Authors          237 non-null    object        
 5   ISBN                        237 non-null    object        
 6   ISBN13                      237 non-null    object        
 7   My Rating                   237 non-null    int64         
 8   Average Rating              237 non-null    float64       
 9   Publisher                   233 non-null    object        
 10  Binding                     237 non-null    object        
 11  Number of Pages             237 non-null    int64         

# Export Data

In [50]:
lib.export_data(export_clean_directory / 'my_books_data.csv', df_cp1)
lib.export_data(export_clean_directory / 'my_books_categories.csv', df_categories)

PortfolioLogger.lib.tools: INFO: Exporting 6873 elements (0.62 MB) to /Users/christophermagno/Documents/Projects/Data Analyst/christophermagno/Projects/My Bookshelf/clean/my_books_data_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10ef5a660> took 0.012 secs to complete.
PortfolioLogger.lib.tools: INFO: Exporting 4078 elements (0.16 MB) to /Users/christophermagno/Documents/Projects/Data Analyst/christophermagno/Projects/My Bookshelf/clean/my_books_categories_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10ef5a660> took 0.003 secs to complete.


PosixPath('/Users/christophermagno/Documents/Projects/Data Analyst/christophermagno/Projects/My Bookshelf/clean/my_books_categories_CLEAN.csv')